In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
!pip install streamlit split-folders wandb -q

     |████████████████████████████████| 9.7 MB 4.2 MB/s 
     |████████████████████████████████| 1.7 MB 38.7 MB/s 
     |████████████████████████████████| 76 kB 4.9 MB/s 
     |████████████████████████████████| 164 kB 46.0 MB/s 
     |████████████████████████████████| 111 kB 52.7 MB/s 
     |████████████████████████████████| 4.3 MB 36.4 MB/s 
     |████████████████████████████████| 180 kB 50.5 MB/s 
     |████████████████████████████████| 63 kB 1.7 MB/s 
     |████████████████████████████████| 128 kB 49.8 MB/s 
     |████████████████████████████████| 792 kB 34.3 MB/s 
     |████████████████████████████████| 380 kB 49.4 MB/s 
     |████████████████████████████████| 144 kB 36.8 MB/s 
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-console 5.2.0 requires prompt-toolkit<2.0.0,>=1.0.0, but you have prompt-toolkit 3.0.28 which is incompatible.
google-colab 1.

In [ ]:
!wandb login #b9000f4b617392245760fc528ade5dbf683930b9

wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [ ]:
import os 
import zipfile 
import tensorflow as tf 
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from tensorflow.keras import layers 
from tensorflow.keras import Model 
import matplotlib.pyplot as plt
import streamlit
from PIL import Image
import numpy as np
import splitfolders
import wandb
from wandb.keras import WandbCallback
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
wandb.init(project="cv_retail_disruption", entity="lakshya2k")

wandb: Currently logged in as: lakshya2k (use `wandb login --relogin` to force relogin)


In [ ]:
wandb.config = {
  "learning_rate": 0.0001,
  "epochs": 100,
  "batch_size": 32,
  "loss": "binary_crossentropy",
  "metrics": ["accuracy"],
  "split_dataset_seed": 42
}
config = wandb.config

In [ ]:
base_dir = "/content/gdrive/MyDrive/Data"
dataset_dir = os.path.join(base_dir, "dataset")
split_dataset_dir = os.path.join(base_dir, "split_dataset")
train_dir = os.path.join(split_dataset_dir, "train")
validation_dir = os.path.join(split_dataset_dir, "val")

In [ ]:
input_folder = dataset_dir
output_folder = split_dataset_dir
splitfolders.ratio(input_folder, output_folder, ratio=(0.7,0.3), seed=config["split_dataset_seed"])

Copying files: 136 files [01:22,  1.65 files/s]


In [ ]:
# Directory with our training open pictures
train_open_dir = os.path.join(train_dir, 'open')

# Directory with our training closed pictures
train_closed_dir = os.path.join(train_dir, 'closed')

# Directory with our validation open pictures
validation_open_dir = os.path.join(validation_dir, 'open')

# Directory with our validation closed pictures
validation_closed_dir = os.path.join(validation_dir, 'closed')

In [ ]:
# Add our data-augmentation parameters to ImageDataGenerator
train_datagen = ImageDataGenerator(rescale = 1./255.,
                                   rotation_range = 30,
                                   width_shift_range = 0.2,
                                   height_shift_range = 0.2,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True,
                                   fill_mode = "reflect"
)

# Note that the validation data should not be augmented!
test_datagen = ImageDataGenerator( rescale = 1.0/255. )

In [ ]:
# Flow training images in batches of 20 using train_datagen generator
train_generator = train_datagen.flow_from_directory(train_dir, batch_size = config["batch_size"], class_mode = 'binary', target_size = (150, 150))

# Flow validation images in batches of 20 using test_datagen generator
validation_generator = test_datagen.flow_from_directory( validation_dir,  batch_size = config["batch_size"], class_mode = 'binary', target_size = (150, 150))

Found 94 images belonging to 2 classes.
Found 42 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3
base_model = InceptionV3(input_shape = (150, 150, 3), include_top = False, weights = 'imagenet')

for layer in base_model.layers:
    layer.trainable = False

   16384/87910968 [..............................] - ETA: 1s

In [ ]:
x = layers.Flatten()(base_model.output)
x = layers.Dense(1024, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)

# Add a final sigmoid layer with 1 node for classification output
x = layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.models.Model(base_model.input, x)

model.compile(optimizer = tf.keras.optimizers.Adam(lr=config["learning_rate"]), loss = config["loss"], metrics = config["metrics"])

In [ ]:
history = model.fit_generator(train_generator,
                              validation_data = validation_generator,
                              epochs = config["epochs"], 
                              callbacks = [WandbCallback(),
                                           EarlyStopping(monitor="val_accuracy", patience=20, min_delta=0.001, mode="max"),
                                           ReduceLROnPlateau(monitor="val_loss", patience=20, min_delta=0.001, mode="min", factor=0.2, min_lr=1e-5),
                                           ]
)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:6: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  


Epoch 1/100
3/3 [==============================] - 10s 2s/step - loss: 1.2286 - accuracy: 0.5638 - val_loss: 0.4865 - val_accuracy: 0.7857 - lr: 1.0000e-04
Epoch 2/100
3/3 [==============================] - 2s 741ms/step - loss: 0.7256 - accuracy: 0.6702 - val_loss: 0.5960 - val_accuracy: 0.6905 - lr: 1.0000e-04
Epoch 3/100
3/3 [==============================] - 3s 1s/step - loss: 0.7513 - accuracy: 0.7021 - val_loss: 0.3167 - val_accuracy: 0.8571 - lr: 1.0000e-04
Epoch 4/100
3/3 [==============================] - 3s 1s/step - loss: 0.6149 - accuracy: 0.8191 - val_loss: 0.3131 - val_accuracy: 0.9048 - lr: 1.0000e-04
Epoch 5/100
3/3 [==============================] - 3s 1s/step - loss: 0.5596 - accuracy: 0.8085 - val_loss: 0.2535 - val_accuracy: 0.9048 - lr: 1.0000e-04
Epoch 6/100
3/3 [==============================] - 2s 751ms/step - loss: 0.6553 - accuracy: 0.7128 - val_loss: 0.2831 - val_accuracy: 0.8810 - lr: 1.0000e-04
Epoch 7/100
3/3 [==============================] - 2s 758ms/ste

In [ ]:
model.save(os.path.join(base_dir,"shop_clf_9362_9524_aug.h5"))

In [ ]:
image = Image.open("closed/11.PNG")
# image = image.resize((150,150))
# image = image.convert("RGB")
# image.size
# image = tf.keras.preprocessing.image.img_to_array(image)
# image.shape
# np.expand_dims(image, axis=0).shape

In [ ]:
def predict(image):
    classifier_model = "shop_clf.h5"
    IMAGE_SHAPE = (150, 150,3)
    model = tf.keras.models.load_model(classifier_model, compile=False)
    test_image = image.resize((150,150)).convert("RGB")
    test_image = tf.keras.preprocessing.image.img_to_array(test_image)
    test_image = test_image / 255.0
    test_image = np.expand_dims(test_image, axis=0)
    predictions = model.predict(test_image)

    # result = f"{class_names[np.argmax(scores)]} with a { (100 * np.max(scores)).round(2) } % confidence." 
    result = [ "This is an Open Shop" if predictions[0][0] >= 0.5 else "This is a closed shop"]
    return result[0]

In [ ]:
predict(image)

'This is a closed shop'

In [ ]:
def app():
    st.title('Shop Classifier')
    st.markdown("Welcome to this simple web application that classifies shops. The shops are classified into two different classes namely: open and closed.")
    file_uploaded = st.file_uploader("Choose File", type=["png","jpg","jpeg"])
    class_btn = st.button("Classify")
    if file_uploaded is not None:    
        image = Image.open(file_uploaded)
        st.image(image, caption='Uploaded Image', use_column_width=True)
        
    if class_btn:
        if file_uploaded is None:
            st.write("Invalid command, please upload an image")
        else:
            with st.spinner('Model working....'):
                plt.imshow(image)
                plt.axis("off")
                predictions = predict(image)
                time.sleep(1)
                st.success('Classified')
                st.write(predictions)
                st.pyplot(fig)

In [ ]:
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip ngrok-stable-linux-amd64.zip
!./ngrok authtoken 24sKQx30BWyJg2vt39TM2mqMhSc_VfgCrgZqgwctUimBtWe4

--2022-02-09 13:37:38--  https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 54.161.241.46, 52.202.168.65, 54.237.133.81, ...
Connecting to bin.equinox.io (bin.equinox.io)|54.161.241.46|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13832437 (13M) [application/octet-stream]
Saving to: ‘ngrok-stable-linux-amd64.zip.1’

ngrok-stable-linux- 100%[===================>]  13.19M  3.36MB/s    in 6.1s    

2022-02-09 13:37:45 (2.16 MB/s) - ‘ngrok-stable-linux-amd64.zip.1’ saved [13832437/13832437]

Archive:  ngrok-stable-linux-amd64.zip
replace ngrok? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ngrok                   
Authtoken saved to configuration file: /root/.ngrok2/ngrok.yml


In [ ]:
get_ipython().system_raw('./ngrok http 8501 &')

In [ ]:
!curl -s http://localhost:4040/api/tunnels | python3 -c \
    'import sys, json; print("Execute the next cell and the go to the following URL: " +json.load(sys.stdin)["tunnels"][0]["public_url"])'

Execute the next cell and the go to the following URL: http://a3ca-35-204-24-137.ngrok.io


In [ ]:
!streamlit run app.py --server.enableWebsocketCompression=false

2022-02-09 13:55:35.709 INFO    numexpr.utils: NumExpr defaulting to 2 threads.

  You can now view your Streamlit app in your browser.

  Network URL: http://172.28.0.2:8501
  External URL: http://35.204.24.137:8501

2022-02-09 13:56:14.675470: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2022-02-09 13:58:13.520 5 out of the last 5 calls to <function Model.make_predict_function.<locals>.predict_function at 0x7f4a7cc0fd40> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has experimental_relax_shapes=True option that relaxes argument shapes that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflo

In [ ]:
import streamlit as st
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow_hub as hub
import tensorflow as tf
import numpy as np
from tensorflow import keras
from tensorflow.keras.models import load_model
from tensorflow.keras import preprocessing
import time

fig = plt.figure()

def app():
    st.title('Shop Classifier')
    st.markdown("Welcome to this simple web application that classifies shops. The shops are classified into two different classes namely: open and closed.")
    file_uploaded = st.file_uploader("Choose File", type=["png","jpg","jpeg"])
    class_btn = st.button("Classify")
    if file_uploaded is not None:    
        image = Image.open(file_uploaded)
        st.image(image, caption='Uploaded Image', use_column_width=True)
        
    if class_btn:
        if file_uploaded is None:
            st.write("Invalid command, please upload an image")
        else:
            with st.spinner('Model working....'):
                plt.imshow(image)
                plt.axis("off")
                predictions = predict(image)
                time.sleep(1)
                st.success('Classified')
                st.write(predictions)
                # st.pyplot(fig)
def predict(image):
    classifier_model = "/content/gdrive/MyDrive/Data/shop_clf.h5"
    IMAGE_SHAPE = (150, 150,3)
    model = tf.keras.models.load_model(classifier_model, compile=False)
    test_image = image.resize((150,150)).convert("RGB")
    test_image = tf.keras.preprocessing.image.img_to_array(test_image)
    test_image = test_image / 255.0
    test_image = np.expand_dims(test_image, axis=0)
    predictions = model.predict(test_image)

    # result = f"{class_names[np.argmax(scores)]} with a { (100 * np.max(scores)).round(2) } % confidence." 
    # result = [ "This is an Open Shop" if predictions[0][0] >= 0.5 else "This is a closed shop"]
    prob = predictions[0][0]
    result = [ f"Shop->Open | Confidence->{prob}" if predictions[0][0] >= 0.5 else f"Shop->Closed | Confidence->{1-prob}"]
    # return f"Probability for open shop -> {prob}"
    return result[0]
if __name__=="__main__":
    app()